# DistilBERT GPU throughput pilot (not full training)

This notebook **does not fine-tune the final model**. The earlier token-length analysis has already compared 128 and 256 tokens and provisionally selected 256 from text coverage. This notebook benchmarks short training loops on the actual Colab GPU, confirms whether that choice fits the eight-hour budget, and selects a physical batch size.

Before running all cells:

1. In Colab, choose **Runtime > Change runtime type > GPU**.
2. Commit and push the repository version that contains this notebook.
3. Upload the matching `training_dataset.parquet` to `MyDrive/AIDI_artefact/private/`.

The notebook only reads training rows during the GPU benchmark. It writes aggregate JSON evidence to the private Drive folder and never commits or pushes changes. Package pins are installed in an isolated virtual environment, so no runtime restart is required. Cells are safe to run again in the same runtime.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

if shutil.which('nvidia-smi') is None:
    raise RuntimeError(
        'No NVIDIA GPU runtime was found. In Colab choose Runtime > Change runtime type > GPU, then run again.'
    )
gpu_check = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True, check=False,
)
if gpu_check.returncode != 0:
    raise RuntimeError('The Colab GPU is not ready:\n' + gpu_check.stderr.strip())
print('GPU:', gpu_check.stdout.strip())

In [ ]:
try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError('This notebook must be run in Google Colab.') from error

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/MurrayMint7/AIDI_artefact.git'
REPO_ROOT = Path('/content/AIDI_artefact')
PRIVATE_ROOT = Path('/content/drive/MyDrive/AIDI_artefact/private')
DATASET_PATH = PRIVATE_ROOT / 'training_dataset.parquet'
MODEL_CACHE = PRIVATE_ROOT / 'huggingface-cache'
EVIDENCE_DIR = PRIVATE_ROOT / 'evidence'
VENV_ROOT = Path('/content/aidi-distilbert-venv')

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_PATH}. Upload the governed Parquet file there, then rerun this cell.'
    )
print('Dataset:', DATASET_PATH)

In [ ]:
if REPO_ROOT.exists():
    if not (REPO_ROOT / '.git').is_dir():
        raise RuntimeError(f'{REPO_ROOT} exists but is not a Git clone. Rename it or delete the Colab runtime.')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    print(f'Updated existing clone at {REPO_ROOT}')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)

required_files = [
    REPO_ROOT / 'requirements-colab.txt',
    REPO_ROOT / 'requirements-transformer.txt',
    REPO_ROOT / 'config/distilbert.yaml',
    REPO_ROOT / 'artifacts/metrics/distilbert_token_length_summary.json',
]
missing = [str(path.relative_to(REPO_ROOT)) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError('The clone is missing required file(s): ' + ', '.join(missing))
commit = subprocess.run(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'],
    capture_output=True, text=True, check=True,
).stdout.strip()
print('Repository commit:', commit)

In [ ]:
# Keep project pins separate from Colab's managed Python environment.
if not (VENV_ROOT / 'bin/python').is_file():
    subprocess.run(
        [sys.executable, '-m', 'venv', '--system-site-packages', str(VENV_ROOT)],
        check=True,
    )
VENV_PYTHON = str(VENV_ROOT / 'bin/python')

subprocess.run(
    [VENV_PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check', '-q',
     '-r', str(REPO_ROOT / 'requirements-colab.txt')],
    check=True,
)
subprocess.run(
    [VENV_PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check', '-q',
     '-e', str(REPO_ROOT), '--no-deps'],
    check=True,
)
print('Isolated environment ready:', VENV_PYTHON)

In [ ]:
# Validate imports, CUDA visibility, Parquet schema, and the dataset/token-summary pairing before downloading the model.
preflight_code = r'''
import hashlib
import importlib.metadata
import json
import sys
from pathlib import Path

import pyarrow.parquet as pq
import torch

dataset_path = Path(sys.argv[1])
summary_path = Path(sys.argv[2])
required_columns = {'text', 'sentiment_label', 'source_category', 'split'}
columns = set(pq.ParquetFile(dataset_path).schema_arrow.names)
missing_columns = sorted(required_columns - columns)
if missing_columns:
    raise RuntimeError(f'Dataset is missing required columns: {missing_columns}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
dataset_hash = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
expected_hash = summary.get('evidence', {}).get('dataset_sha256')
if dataset_hash != expected_hash:
    raise RuntimeError(
        'The Drive dataset does not match the token-length summary in this repository. '
        'Upload the exact training_dataset.parquet used for the token-length pilot.'
    )
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot see CUDA. Select a GPU runtime and rerun from the first cell.')
for package in ['torch', 'transformers', 'pandas', 'pyarrow', 'fsspec']:
    print(package, importlib.metadata.version(package))
print('CUDA device:', torch.cuda.get_device_name(0))
print('Dataset and token summary match:', dataset_hash)
'''
subprocess.run(
    [VENV_PYTHON, '-c', preflight_code, str(DATASET_PATH),
     str(REPO_ROOT / 'artifacts/metrics/distilbert_token_length_summary.json')],
    cwd=REPO_ROOT, check=True,
)

In [ ]:
command = [
    VENV_PYTHON, '-m', 'amazon_sentiment', 'throughput',
    '--config', 'config/distilbert.yaml',
    '--dataset', str(DATASET_PATH),
    '--token-summary', 'artifacts/metrics/distilbert_token_length_summary.json',
    '--output-dir', 'artifacts/metrics',
    '--cache-dir', str(MODEL_CACHE),
]
subprocess.run(command, cwd=REPO_ROOT, check=True)
print('GPU throughput pilot completed.')

In [ ]:
import json

EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
output_names = [
    'distilbert_throughput_results.json',
    'distilbert_pilot_decision.json',
    'distilbert_colab_environment.json',
]
for name in output_names:
    source = REPO_ROOT / 'artifacts/metrics' / name
    if not source.is_file():
        raise FileNotFoundError(f'Expected pilot output was not created: {source}')
    shutil.copy2(source, EVIDENCE_DIR / name)

decision_path = EVIDENCE_DIR / 'distilbert_pilot_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print(json.dumps(decision, indent=2))
print('Evidence copied to:', EVIDENCE_DIR)